## Red Wine Quality - 로지스틱 회귀 & SVM으로 개선 시도

07에서 5개 모델 (KNN·NB·트리·RF·XGBoost)을 비교했고,
좋은 와인 f1 기준은 **GaussianNB (0.50)**가 Best.
여기서는 로지스틱 회귀와 SVM을 추가로 검증한다.

In [2]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

df = pd.read_csv("../data/redwine_quality.csv")
X = df.drop(columns=["quality"])
y_bin = (df["quality"] >= 7).astype(int)

print(X.shape, "좋은 와인 비율:", y_bin.mean().round(3))

(1599, 11) 좋은 와인 비율: 0.136


In [3]:
models = {
    # 로지스틱 회귀:
    "LogisticRegression": Pipeline([
        ("sc", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000)),
    ]),
    # SVM
    "SVM": Pipeline([
        ("sc", StandardScaler()),
        ("svm", SVC()),
    ]),
    "LogisticRegression(balanced)": Pipeline([
        ("sc", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    # SVM
    "SVM(balanced)": Pipeline([
        ("sc", StandardScaler()),
        ("svm", SVC(class_weight="balanced")),
    ]),
}

In [6]:
for name, model in models.items():
    f1 = cross_val_score(model, X, y_bin, cv=5, scoring="f1")
    print(f"{name:30s} | f1 {f1.mean():.4f}")
print("-" * 45)
print(f"{'(07 기준) GaussianNB':30s} | f1 0.4976")

LogisticRegression             | f1 0.3554
SVM                            | f1 0.2955
LogisticRegression(balanced)   | f1 0.5177
SVM(balanced)                  | f1 0.5156
---------------------------------------------
(07 기준) GaussianNB             | f1 0.4976


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_bin, test_size=0.2, stratify=y_bin, random_state=42
)

# balanced 두 모델을 test로 비교
for name in ["LogisticRegression(balanced)", "SVM(balanced)"]:
    model = models[name]
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("=" * 50)
    print(name)
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=["나쁨", "좋음"], digits=3))


LogisticRegression(balanced)
[[215  62]
 [  7  36]]
              precision    recall  f1-score   support

          나쁨      0.968     0.776     0.862       277
          좋음      0.367     0.837     0.511        43

    accuracy                          0.784       320
   macro avg      0.668     0.807     0.686       320
weighted avg      0.888     0.784     0.815       320

SVM(balanced)
[[233  44]
 [  9  34]]
              precision    recall  f1-score   support

          나쁨      0.963     0.841     0.898       277
          좋음      0.436     0.791     0.562        43

    accuracy                          0.834       320
   macro avg      0.699     0.816     0.730       320
weighted avg      0.892     0.834     0.853       320



### CV 결과 (cv=5, 좋은와인 f1)
| 모델 | f1 |
|---|---|
| LogisticRegression (기본) | 0.355 |
| SVM (기본) | 0.296 |
| LogisticRegression (balanced) | 0.518 |
| SVM (balanced) | 0.516 |
| (07) GaussianNB | 0.498 |

→ 기본 모델은 GaussianNB보다 낮음. balanced가 결정적.

### test 최종 비교 (좋은 와인 기준)
| 모델 | precision | recall | f1 |
|---|---|---|---|
| GaussianNB (07) | 0.45 | 0.67 | 0.54 |
| LogisticRegression (bal) | 0.37 | **0.84** | 0.51 |
| SVM (balanced) | 0.44 | 0.79 | **0.56** |

In [9]:
from sklearn.model_selection import GridSearchCV

# SVM의 핵심 하이퍼파라미터 격자
param_grid = {
    "svm__C": [0.1, 1, 10, 100],           # 규제 강도 (클수록 train에 맞춤)
    "svm__gamma": ["scale", 0.01, 0.1, 1], # 커널 폭 (경계의 유연성)
    "svm__kernel": ["rbf"],                 # 비선형 커널
}

# Pipeline에 balanced SVM (파라미터명은 "단계이름__파라미터")
svm_pipe = Pipeline([
    ("sc", StandardScaler()),
    ("svm", SVC(class_weight="balanced")),
])

grid = GridSearchCV(
    svm_pipe,
    param_grid,
    scoring="f1",       # ← 좋은와인 f1로 최고를 고름 (우리 목적!)
    cv=5,
    n_jobs=-1,          # 병렬로 빠르게
)
grid.fit(X, y_bin)

print("최고 파라미터:", grid.best_params_)
print("최고 CV f1:", round(grid.best_score_, 4))
print("(기본 SVM balanced f1: 0.5156)")


최고 파라미터: {'svm__C': 1, 'svm__gamma': 0.1, 'svm__kernel': 'rbf'}
최고 CV f1: 0.5228
(기본 SVM balanced f1: 0.5156)


In [10]:
# GridSearchCV가 찾은 최적 모델은 grid.best_estimator_에 이미 전체 학습됨
# 공정한 test 평가 위해 train/test 다시 나눠 재학습
from sklearn.metrics import classification_report, confusion_matrix

best_svm = grid.best_estimator_          # 최적 파라미터로 세팅된 파이프라인
best_svm.fit(X_train, y_train)           # train으로 학습
pred = best_svm.predict(X_test)

print("튜닝된 SVM (C=1, gamma=0.1)")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["나쁨", "좋음"], digits=3))
print("(튜닝 전 SVM test f1: 0.562)")


튜닝된 SVM (C=1, gamma=0.1)
[[233  44]
 [  9  34]]
              precision    recall  f1-score   support

          나쁨      0.963     0.841     0.898       277
          좋음      0.436     0.791     0.562        43

    accuracy                          0.834       320
   macro avg      0.699     0.816     0.730       320
weighted avg      0.892     0.834     0.853       320

(튜닝 전 SVM test f1: 0.562)


### 핵심 발견 4: GridSearchCV — 튜닝이 항상 개선은 아니다
- 최적 파라미터 C=1, gamma=0.1 → 사실상 기본값. CV f1 0.5228 (거의 그대로).
- test 결과는 튜닝 전과 완전히 동일 (f1 0.562).
- 교훈: 큰 개선은 balanced(불균형 대응)에서 왔고, 튜닝은 미세 조정.
  sklearn 기본값이 이미 좋고, 데이터 한계는 튜닝으로 못 넘는다.
